# Connecting to the Prompt Hub

We can connect our application to LangSmith's Prompt Hub, which will allow us to test and iterate on our prompts within LangSmith, and pull our improvements directly into our application.

### Setup

### Edit! API keys etc. from Codespace secrets (straight to env variables)

In [ ]:
#import os
#os.environ["OPENAI_API_KEY"] = ""
#os.environ["LANGCHAIN_API_KEY"] = ""
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_PROJECT"] = "langsmith-academy"  # If you don't set this, traces will go to the Default project

In [ ]:
# Or you can use a .env file
#from dotenv import load_dotenv
#load_dotenv(dotenv_path="../../.env", override=True)

### Pull a prompt from Prompt Hub

Pull in a prompt from Prompt Hub by pasting in the code snippet from the UI.

In [1]:
# Create a LANGSMITH_API_KEY in Settings > API Keys
from langsmith import Client
client = Client() # API key defaults to env variable
prompt = client.pull_prompt("polyglot_pirate")

Let's see what we pulled - note that we did not get the model, so this is just a StructuredPrompt and not runnable.

In [2]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'polyglot_pirate', 'lc_hub_commit_hash': '6d44b2edfe8a4d13acd3b854d7e8fc3b8b57534207b0b0cd36c0daefdef6ab6c'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from the 1600s. You only speak {language}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extract the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'The answer from to LLM to the user.'}}, 'required': ['answer'], 'strict': True, 'additionalProperties': False}, structured_output_kwargs={})

Cool! Now let's hydrate our prompt by calling .invoke() with our inputs

In [3]:
hydrated_prompt = prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})
hydrated_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a pirate from the 1600s. You only speak Spanish.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Are you a captain yet?', additional_kwargs={}, response_metadata={})])

And now let's pass those messages to OpenAI and see what we get back!

In [4]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

# NOTE: We can use this utility from LangSmith to convert our hydrated prompt to openai format
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-B7kg41lsszGCQzhMjtuhUycSWi05N', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='¡Ahoy! No soy un capitán, pero soy un pirata astuto y valiente. Busco tesoros y aventuras en alta mar. ¿Qué deseas saber, amigo navegante?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1741187424, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_06737a9306', usage=CompletionUsage(completion_tokens=41, prompt_tokens=33, total_tokens=74, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

##### [Extra: LangChain Only] Pulling down the Model Configuration

We can also pull down the saved model configuration as a LangChain RunnableBinding when we use `include_model=True`. This allows us to run our prompt template directly with the saved model configuration.

In [5]:
prompt = client.pull_prompt("polyglot_pirate", include_model=True)

/usr/local/python/3.12.1/lib/python3.12/json/decoder.py:337: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  obj, end = self.raw_decode(s, idx=_w(s, 0).end())


In [6]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'polyglot_pirate', 'lc_hub_commit_hash': '6d44b2edfe8a4d13acd3b854d7e8fc3b8b57534207b0b0cd36c0daefdef6ab6c'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from the 1600s. You only speak {language}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extract the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'The answer from to LLM to the user.'}}, 'required': ['answer'], 'strict': True, 'additionalProperties': False}, structured_output_kwargs={})
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completio

Test out your prompt!

In [7]:
prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})

{'answer': '¡Aún no soy capitán, pero navego por los mares buscando tesoros y aventuras! Quizás un día pueda ser el capitán de mi propia nave.'}

### Pull down a specific commit

Pull down a specific commit from the Prompt Hub by pasting in the code snippet from the UI.

In [8]:
from langchain import hub
prompt = hub.pull("polyglot_pirate:61b84fd5")

Run this commit!

In [9]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

hydrated_prompt = prompt.invoke({"question": "What is the world like?", "language": "English"})
# NOTE: We can use this utility from LangSmith to convert our hydrated prompt to openai format
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-B7koXwsS7EtGAaYrVkhsAr6CbbAYx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Ahoy, matey! In the year 2500, the world be a vastly differin' place than what ye once knew. Technology be more advanced than a ship's AI navigatin' through a stormy sea. Most landlubbers utilize virtual reality and augmented reality in their daily lives, makin' the real world blend with digital realms.\n\nThe oceans be a mix of pristine waters and carefully managed ecosystems, thanks to advancements in environmental science. Many pirates, like meself, find treasure not just in gold but in renewable resources and data. AI and robotics be workin' alongside humans, and space travel has become as common as settin' sail on the high seas.\n\nSociety be more interconnected than ever, but challenges still abound. There be conflicts over resources, and some territories still be havin' their share of scallywags and brigands. But us pira

### Uploading Prompts

You can also easily update your prompts in the hub programmatically.



In [10]:
from langchain.prompts.chat import ChatPromptTemplate
from langsmith import Client

client=Client()

french_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context} 
Question: {question}
Answer:"""

french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
client.push_prompt("french-rag-prompt", object=french_prompt_template)

'https://smith.langchain.com/prompts/french-rag-prompt/75567b82?organizationId=4c6e2fdf-dbd6-59d4-9ae2-e8c30b5ca048'

You can also push a prompt as a RunnableSequence of a prompt and a model. This is useful for storing the model configuration you want to use with this prompt. The provider must be supported by the LangSmith playground.

In [11]:
from langchain.prompts.chat import ChatPromptTemplate
from langsmith import Client
from langchain_openai import ChatOpenAI

client=Client()
model = ChatOpenAI(model="gpt-4o-mini")

french_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context} 
Question: {question}
Answer:"""
french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
chain = french_prompt_template | model
client.push_prompt("french-runnable-sequence", object=chain)

'https://smith.langchain.com/prompts/french-runnable-sequence/34b9ffab?organizationId=4c6e2fdf-dbd6-59d4-9ae2-e8c30b5ca048'